In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import skew, kurtosis
from scipy.optimize import linprog
from sklearn.covariance import LedoitWolf
import warnings
warnings.filterwarnings('ignore')

class FourTypesPortfolioOptimizer:
    """
    四类投资组合优化:
    1. 原始模型 (Original Models) - 只考虑基本风险度量
    2. 考虑高阶矩的模型 (Higher-Order Moments Models) - 基本风险 + 偏度峰度
    3. 考虑预测收益的模型 (Predictive Information Models) - 基本风险 + 预测信息
    4. 综合模型 (Integrated Models) - 基本风险 + 高阶矩 + 预测信息
    """
    
    def __init__(self, data_file):
        """初始化四类模型优化"""
        self.data_file = data_file
        
        # 数据
        self.raw_data = None
        self.yearly_stocks = {}
        
        # 优化结果 - 四类模型
        self.daily_weights = {}
        self.portfolio_returns = {}
        self.backtest_results = None
        
        # 参数设置
        self.lookback_days = 20
        self.min_weight = 0.01
        self.max_weight = 0.5
        self.confidence_level = 0.05
        self.target_return = 0.0 
        self.transaction_cost_rate = Transaction_cost_rate

        self.base_models = ['MV', 'MSV', 'MAD', 'MSAD', 'CVaR', 'Omega']

        self.model_types = {
            'Original': [],           # 原始模型 - 只考虑基本风险
            'HigherMoments': [],      # 高阶矩模型 - 基本风险 + 偏度峰度
            'Predictive': [],         # 预测信息模型 - 基本风险 + 预测信息
            'Integrated': []          # 综合模型 - 基本风险 + 高阶矩 + 预测信息
        }

        self._generate_model_names()

        self.auxiliary_vars = {
            'tau': 0.0,     # Omega模型阈值
            'delta': 0.5,   # Omega模型风险偏好参数
            'alpha': 0.9    # CVaR置信水平
        }

    def calculate_transaction_cost(self, old_weights, new_weights, old_stocks, new_stocks, portfolio_value=1.0):
        """
        计算交易成本
        
        Parameters:
        -----------
        old_weights : np.array
            上期权重
        new_weights : np.array  
            新期权重
        old_stocks : list
            上期股票代码列表
        new_stocks : list
            新期股票代码列表
        portfolio_value : float
            投资组合总价值，默认为1（表示按比例计算）
        
        Returns:
        --------
        cost_rate : float
            交易成本率（相对于投资组合价值）
        cost_amount : float
            交易成本金额
        transaction_volume : float
            交易额
        turnover_rate : float
            换手率
        """
        # 如果是第一次投资，所有权重都是新买入
        if old_weights is None or old_stocks is None:
            # 首次投资，买入所有股票
            turnover_rate = np.sum(new_weights)  # 应该等于1
            transaction_volume = turnover_rate * portfolio_value
            cost_amount = transaction_volume * self.transaction_cost_rate
            cost_rate = cost_amount / portfolio_value
            
            return cost_rate, cost_amount, transaction_volume, turnover_rate

        aligned_new, aligned_old = self._align_weights(new_stocks, new_weights, old_stocks, old_weights)
        weight_changes = np.abs(aligned_new - aligned_old)
        # 换手率 = 所有权重变化的和
        turnover_rate = np.sum(weight_changes)
        transaction_volume = turnover_rate * portfolio_value
        # 交易成本 = 交易额 × 成本率
        cost_amount = transaction_volume * self.transaction_cost_rate
        # 交易成本率（相对于投资组合价值）
        cost_rate = cost_amount / portfolio_value
        
        return cost_rate, cost_amount, transaction_volume, turnover_rate

    def _align_weights(self, current_stocks, current_weights, previous_stocks, previous_weights):
        """
        对齐不同期间的股票权重，处理股票池变化
        
        Returns:
        --------
        aligned_current : np.array
            对齐后的当期权重
        aligned_previous : np.array
            对齐后的前期权重
        """
        # 获取所有出现过的股票
        all_stocks = list(set(current_stocks + previous_stocks))
        all_stocks.sort() 
        aligned_current = np.zeros(len(all_stocks))
        aligned_previous = np.zeros(len(all_stocks))
        
        # 填入当期权重
        for i, stock in enumerate(current_stocks):
            idx = all_stocks.index(stock)
            aligned_current[idx] = current_weights[i]
        
        # 填入前期权重
        for i, stock in enumerate(previous_stocks):
            idx = all_stocks.index(stock)
            aligned_previous[idx] = previous_weights[i]
        
        return aligned_current, aligned_previous

    def calculate_four_types_returns_with_cost(self):
        """计算考虑交易成本的四类投资组合模型收益"""
        
        portfolio_returns = {}
        
        # 计算所有模型的收益
        all_models = []
        for model_type in self.model_types:
            all_models.extend(self.model_types[model_type])
        
        for model in all_models:
            if model not in self.daily_weights:
                continue
                
            print(f"处理模型: {model}")
            
            daily_returns = []
            previous_weights = None  # 存储上期权重
            previous_stock_codes = None  # 存储上期股票代码
            
            # 按日期排序
            sorted_dates = sorted(self.daily_weights[model].keys())
            
            for i, date in enumerate(sorted_dates):
                if i==1 or i==400 or i==800: 
                    print(f"  处理日期: {date} ({i}/{len(sorted_dates)})")
                    
                weight_info = self.daily_weights[model][date]
                current_stock_codes = weight_info['stock_codes']
                current_weights = weight_info['weights']
                
                # 获取次日收益率
                next_date = date + pd.Timedelta(days=1)
                
                # 计算基础投资收益
                basic_return = 0
                valid_weights = 0
                
                for j, stock_code in enumerate(current_stock_codes):
                    # 查找次日的实际收益率
                    next_day_data = self.raw_data[
                        (self.raw_data['stock_code'] == stock_code) &
                        (self.raw_data['date'] == next_date)
                    ]
                    
                    if not next_day_data.empty:
                        stock_return = next_day_data['actual_return'].iloc[0]
                        if not pd.isna(stock_return):
                            basic_return += current_weights[j] * stock_return
                            valid_weights += current_weights[j]
                
                # 标准化权重
                if valid_weights > 0:
                    basic_return = basic_return / valid_weights
                else:
                    basic_return = 0
                
                # 计算交易成本
                cost_rate, cost_amount, transaction_volume, turnover_rate = self.calculate_transaction_cost(
                    previous_weights, current_weights, 
                    previous_stock_codes, current_stock_codes,
                    portfolio_value=1.0  # 按比例计算
                )
                
                # 计算净收益率（基础收益 - 交易成本）
                net_return = basic_return - cost_rate
                
                daily_returns.append({
                    'date': date,
                    'basic_return': basic_return,           # 基础收益率
                    'transaction_cost_rate': cost_rate,     # 交易成本率
                    'transaction_cost_amount': cost_amount, # 交易成本金额
                    'transaction_volume': transaction_volume, # 交易额
                    'turnover_rate': turnover_rate,         # 换手率
                    'net_return': net_return,               # 净收益率
                    'valid_weights': valid_weights
                })
                
                # 更新前期权重
                previous_weights = current_weights.copy()
                previous_stock_codes = current_stock_codes.copy()
            
            portfolio_returns[model] = pd.DataFrame(daily_returns)
        
        # 保存到类属性
        self.portfolio_returns_with_cost = portfolio_returns
        
        return portfolio_returns

        
    def _generate_model_names(self):
        """生成所有四类模型的名称"""
        for base in self.base_models:
            self.model_types['Original'].append(f"{base}_Original")
            self.model_types['HigherMoments'].append(f"{base}_HigherMoments")
            self.model_types['Predictive'].append(f"{base}_Predictive")
            self.model_types['Integrated'].append(f"{base}_Integrated")
    
    def _format_stock_code(self, code):
        """格式化股票代码为6位数字格式"""
        code_int = int(float(code))
        return f"{code_int:06d}"
    
    def load_data(self):
        """加载数据并分析每年包含的股票"""
        self.raw_data = pd.read_csv(self.data_file)
        self.raw_data['date'] = pd.to_datetime(self.raw_data['date'])
        self.raw_data['stock_code'] = self.raw_data['stock_code'].apply(self._format_stock_code)
        self.raw_data['year'] = self.raw_data['date'].dt.year
        
        self.analyze_yearly_stocks()
        
        print(f"总共 {len(self.raw_data)} 条记录")
        return self.raw_data
    
    def analyze_yearly_stocks(self):
        """分析每年包含的股票列表""" 
        for year in range(2020, 2025):
            year_data = self.raw_data[self.raw_data['year'] == year]
            stocks = year_data['stock_code'].unique()
            self.yearly_stocks[year] = sorted(stocks)
        return self.yearly_stocks
    
    def get_daily_portfolio_stocks(self, date):
        """获取指定日期的投资组合股票"""
        year = date.year
        
        if year not in self.yearly_stocks:
            return []
        
        daily_data = self.raw_data[
            (self.raw_data['date'] == date) & 
            (self.raw_data['year'] == year)
        ].copy()
        
        if daily_data.empty:
            return []
        
        # 按预测收益率排序，取前10只
        daily_data = daily_data.sort_values('predicted_return', ascending=False)
        top_10 = daily_data.head(10)
        
        # 只选择预测收益率大于0的股票
        positive_return_stocks = top_10[top_10['predicted_return'] > 0]
        
        if len(positive_return_stocks) == 0:
            return []
        
        return positive_return_stocks['stock_code'].tolist()
    
    def get_historical_returns(self, stock_codes, end_date):
        """获取股票的历史收益率数据"""
        start_date = end_date - pd.Timedelta(days=self.lookback_days)
        
        returns_data = {}
        portfolio_stats = {}
        
        for stock_code in stock_codes:
            stock_data = self.raw_data[
                (self.raw_data['stock_code'] == stock_code) &
                (self.raw_data['date'] >= start_date) &
                (self.raw_data['date'] <= end_date)
            ].sort_values('date')

            returns = stock_data['actual_return'].dropna()
            returns_data[stock_code] = returns.values
            
            # 计算预测误差
            prediction_errors = []
            for _, row in stock_data.iterrows():
                if not pd.isna(row['actual_return']) and not pd.isna(row['predicted_return']):
                    error = row['actual_return'] - row['predicted_return']
                    prediction_errors.append(error)
            
            mean_prediction_error = np.mean(prediction_errors) if prediction_errors else 0
            
            portfolio_stats[stock_code] = {
                'mean_return': returns.mean(),
                'volatility': returns.std(),
                'skewness': stock_data['skew_20d'].iloc[-1] if 'skew_20d' in stock_data.columns else skew(returns),
                'kurtosis': stock_data['kurt_20d'].iloc[-1] if 'kurt_20d' in stock_data.columns else kurtosis(returns),
                'predicted_return': stock_data['predicted_return'].iloc[-1],
                'mean_prediction_error': mean_prediction_error
            }
        return returns_data, portfolio_stats
    
    def calculate_covariance_matrix(self, returns_data):
        """计算协方差矩阵"""
        stock_codes = list(returns_data.keys())
        
        min_length = min(len(returns) for returns in returns_data.values())
        min_length = max(min_length, 3)
        
        returns_matrix = []
        for stock_code in stock_codes:
            returns = returns_data[stock_code]
            if len(returns) >= min_length:
                returns_matrix.append(returns[-min_length:])
            else:
                mean_ret = np.mean(returns)
                padded_returns = np.concatenate([returns, np.full(min_length - len(returns), mean_ret)])
                returns_matrix.append(padded_returns)
        
        returns_matrix = np.array(returns_matrix).T

        lw = LedoitWolf()
        cov_matrix = lw.fit(returns_matrix).covariance_
        
        eigenvals = np.linalg.eigvals(cov_matrix)
        if np.min(eigenvals) <= 0:
            regularization = abs(np.min(eigenvals))
            cov_matrix += np.eye(cov_matrix.shape[0]) * regularization
        
        return cov_matrix
        
# ========== 1. 原始模型 (Original Models) - 只考虑基本风险度量 ==========
    
    def mv_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MV原始模型 - 最小化方差"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        def objective_function(weights):
            # min Σwi*wj*σij
            variance_term = np.dot(weights, np.dot(cov_matrix, weights))
            return variance_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MV')
    
    def msv_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSV原始模型 - 最小化半方差"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        # 计算半方差矩阵
        semivariance_matrix = self._calculate_semivariance_matrix(stock_codes, returns_data, cov_matrix)
        
        def objective_function(weights):
            # min Σwi*wj*σ-ij
            semivariance_term = np.dot(weights, np.dot(semivariance_matrix, weights))
            return semivariance_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MSV')
    
    def mad_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MAD原始模型 - 最小化平均绝对偏差"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        # MAD模型需要线性化，使用辅助变量dt
        def mad_linear_optimization():
            """使用线性规划求解MAD模型"""
            
            
            # 构建线性规划问题
            # 变量: [w1, w2, ..., wn, d1, d2, ..., dT]
            # 目标: min (1/T) * Σdt
            
            # 目标函数系数
            c = np.zeros(n + T)
            c[n:] = 1.0 / T  # dt的系数
            
            # 不等式约束: dt >= Σ(r̂i - rit)*wi 和 dt >= -Σ(r̂i - rit)*wi
            A_ub = []
            b_ub = []
            
            # 计算预期收益（使用历史均值作为r̂i）
            expected_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi
                constraint1 = np.zeros(n + T)
                constraint1[:n] = -deviation  # -Σ(r̂i - rit)*wi
                constraint1[n + t] = 1  # dt
                A_ub.append(constraint1)
                b_ub.append(0)
                
                # dt >= -Σ(r̂i - rit)*wi
                constraint2 = np.zeros(n + T)
                constraint2[:n] = deviation  # Σ(r̂i - rit)*wi
                constraint2[n + t] = 1  # dt
                A_ub.append(constraint2)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))  # wi
            for t in range(T):
                bounds.append((0, None))  # dt >= 0

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                            A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n


        return mad_linear_optimization()
    
    def msad_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSAD原始模型 - 最小化半绝对偏差"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def msad_linear_optimization():
            """使用线性规划求解MSAD模型"""
            
            # 构建线性规划问题
            # 变量: [w1, w2, ..., wn, d1, d2, ..., dT]
            # 目标: min (1/T) * Σdt
            
            # 目标函数系数
            c = np.zeros(n + T)
            c[n:] = 1.0 / T  # dt的系数
            
            # 不等式约束: dt >= Σ(r̂i - rit)*wi 和 dt >= 0
            A_ub = []
            b_ub = []
            
            # 计算预期收益
            expected_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi (只考虑负偏差)
                constraint = np.zeros(n + T)
                constraint[:n] = -deviation
                constraint[n + t] = 1
                A_ub.append(constraint)
                b_ub.append(0)
            
            # dt >= 0 约束已经在bounds中设置
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        
        return msad_linear_optimization()
    
    def cvar_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """CVaR原始模型 - 最小化条件风险价值"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        alpha = self.auxiliary_vars['alpha']  # 0.9
        
        def cvar_linear_optimization():
            """使用线性规划求解CVaR模型"""
            
            # 变量: [w1, w2, ..., wn, ξ, η1, η2, ..., ηT]
            # 目标: min ξ + 1/(1-α) * (1/T) * Σηt
            
            # 目标函数系数
            c = np.zeros(n + 1 + T)
            c[n] = 1.0  # ξ的系数
            c[n+1:] = 1.0 / ((1 - alpha) * T)  # ηt的系数
            
            # 不等式约束: ηt >= -Σwi*rit - ξ, ηt >= 0
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                
                # ηt >= -Σwi*rit - ξ
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t  # Σwi*rit
                constraint[n] = 1  # ξ
                constraint[n + 1 + t] = -1  # -ηt
                A_ub.append(constraint)
                b_ub.append(0)
            
            # ηt >= 0 约束在bounds中设置
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))  # wi
            bounds.append((None, None))  # ξ
            for t in range(T):
                bounds.append((0, None))  # ηt >= 0

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        
        return cvar_linear_optimization()
    
    def omega_original_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """Omega原始模型 - 最大化Omega比率"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        tau = self.auxiliary_vars['tau']  # 0.0
        delta = self.auxiliary_vars['delta']  # 0.5
        
        def omega_linear_optimization():
            """使用线性规划求解Omega模型"""
            
            # 变量: [w1, w2, ..., wn, ψ, η1, η2, ..., ηT]
            # 目标: max ψ (转换为 min -ψ)
            
            # 目标函数系数
            c = np.zeros(n + 1 + T)
            c[n] = -1.0  # -ψ (最大化ψ)
            
            # 不等式约束:
            # δ*(Σwi*r̄i - τ) - (1-δ)/T * Σηt >= ψ
            # ηt >= -Σwi*rit + τ, ηt >= 0
            A_ub = []
            b_ub = []
            
            # 计算历史平均收益
            mean_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            # 主约束: δ*(Σwi*r̄i - τ) - (1-δ)/T * Σηt >= ψ
            # 转换为: -δ*Σwi*r̄i + (1-δ)/T * Σηt + ψ <= -δ*τ
            main_constraint = np.zeros(n + 1 + T)
            main_constraint[:n] = -delta * mean_returns  # -δ*Σwi*r̄i
            main_constraint[n] = 1  # ψ
            main_constraint[n+1:] = (1 - delta) / T  # (1-δ)/T * Σηt
            A_ub.append(main_constraint)
            b_ub.append(-delta * tau)
            
            # ηt约束: ηt >= -Σwi*rit + τ
            for t in range(T):
                returns_t = returns_matrix[:, t]
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t  # Σwi*rit
                constraint[n + 1 + t] = -1  # -ηt
                A_ub.append(constraint)
                b_ub.append(tau)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))  # wi
            bounds.append((None, None))  # ψ
            for t in range(T):
                bounds.append((0, None))  # ηt >= 0

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        return omega_linear_optimization()

    def _solve_optimization_with_linear_constraints(self, objective_function, n, model_type):
        """
        通用优化求解器 - 只有基本的权重约束
        约束条件:
        1. Σwi = 1 (权重和为1)
        2. 0.01 ≤ wi ≤ 0.5 (单个权重限制)
        """
        if n == 0:
            return np.array([])
        
        if n == 1:
            return np.array([1.0])
        
        # 基本约束：权重和为1
        constraints = [
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        ]
        
        # 权重边界：1% - 50%
        bounds = [(self.min_weight, self.max_weight) for _ in range(n)]
        
        # 初始权重：等权重
        x0 = np.ones(n) / n
        
        try:
            result = minimize(
                objective_function,
                x0,
                method='SLSQP',
                bounds=bounds,
                constraints=constraints,
                options={'maxiter': 1000, 'ftol': 1e-9}
            )
            
            if result.success:
                return result.x
            else:
                return np.ones(n) / n
        except Exception as e:
            return np.ones(n) / n
        
# ========== 2. 考虑高阶矩的模型 (Higher-Order Moments Models) ==========
    # 在原始模型基础上加入偏度和峰度考虑
    
    def mv_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MV + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        
        def objective_function(weights):
            # 基本方差项
            variance_term = np.dot(weights, np.dot(cov_matrix, weights))
            # 偏度项 (最大化偏度，所以为负)
            skewness_term = np.sum(weights * individual_skewness)
            # 峰度项 (最小化峰度)
            kurtosis_term = np.sum(weights * individual_kurtosis)
            
            # min variance - skewness + kurtosis
            return variance_term - skewness_term + kurtosis_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MV_HigherMoments')
    
    def msv_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSV + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        semivariance_matrix = self._calculate_semivariance_matrix(stock_codes, returns_data, cov_matrix)
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        
        def objective_function(weights):
            semivariance_term = np.dot(weights, np.dot(semivariance_matrix, weights))
            skewness_term = np.sum(weights * individual_skewness)
            kurtosis_term = np.sum(weights * individual_kurtosis)
            
            return semivariance_term - skewness_term + kurtosis_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MSV_HigherMoments')
    
    def mad_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MAD + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def mad_higher_moments_linear_optimization():
            """MAD + 高阶矩的线性规划求解"""
            
            # 变量: [w1, w2, ..., wn, d1, d2, ..., dT]
            # 目标: min (1/T) * Σdt - Σwi*skewi + Σwi*kurti
            
            # 目标函数系数
            c = np.zeros(n + T)
            c[:n] = -individual_skewness + individual_kurtosis  # 高阶矩项
            c[n:] = 1.0 / T  # MAD项
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            expected_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi
                constraint1 = np.zeros(n + T)
                constraint1[:n] = -deviation
                constraint1[n + t] = 1
                A_ub.append(constraint1)
                b_ub.append(0)
                
                # dt >= -Σ(r̂i - rit)*wi
                constraint2 = np.zeros(n + T)
                constraint2[:n] = deviation
                constraint2[n + t] = 1
                A_ub.append(constraint2)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return mad_higher_moments_linear_optimization()
    
    def msad_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSAD + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def msad_higher_moments_linear_optimization():
            """MSAD + 高阶矩的线性规划求解"""
            
            # 目标函数系数
            c = np.zeros(n + T)
            c[:n] = -individual_skewness + individual_kurtosis
            c[n:] = 1.0 / T
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            expected_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi (只考虑负偏差)
                constraint = np.zeros(n + T)
                constraint[:n] = -deviation
                constraint[n + t] = 1
                A_ub.append(constraint)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return msad_higher_moments_linear_optimization()
    
    def cvar_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """CVaR + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        alpha = self.auxiliary_vars['alpha']
        
        def cvar_higher_moments_linear_optimization():
            """CVaR + 高阶矩的线性规划求解"""
            
            # 变量: [w1, w2, ..., wn, ξ, η1, η2, ..., ηT]
            # 目标: min ξ + 1/(1-α)*(1/T)*Σηt - Σwi*skewi + Σwi*kurti
            
            c = np.zeros(n + 1 + T)
            c[:n] = -individual_skewness + individual_kurtosis  # 高阶矩项
            c[n] = 1.0  # ξ
            c[n+1:] = 1.0 / ((1 - alpha) * T)  # ηt
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                
                # ηt >= -Σwi*rit - ξ
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n] = 1
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ξ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return cvar_higher_moments_linear_optimization()
    
    def omega_higher_moments_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """Omega + 高阶矩模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        tau = self.auxiliary_vars['tau']
        delta = self.auxiliary_vars['delta']
        
        def omega_higher_moments_linear_optimization():
            """Omega + 高阶矩的线性规划求解"""
            
            # 变量: [w1, w2, ..., wn, ψ, η1, η2, ..., ηT]
            # 目标: max ψ + Σwi*skewi - Σwi*kurti (转换为min)
            
            c = np.zeros(n + 1 + T)
            c[:n] = individual_skewness - individual_kurtosis  # 高阶矩项(取负)
            c[n] = -1.0  # -ψ (最大化ψ)
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            mean_returns = np.array([np.mean(returns_data[code]) for code in stock_codes])
            
            # 主约束修改为包含高阶矩
            main_constraint = np.zeros(n + 1 + T)
            main_constraint[:n] = -delta * mean_returns
            main_constraint[n] = 1  # ψ
            main_constraint[n+1:] = (1 - delta) / T
            A_ub.append(main_constraint)
            b_ub.append(-delta * tau)
            
            # ηt约束
            for t in range(T):
                returns_t = returns_matrix[:, t]
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(tau)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ψ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return omega_higher_moments_linear_optimization()
    
# ========== 3. 考虑预测收益的模型 (Predictive Information Models) ==========
    # 在原始模型基础上加入预测收益和预测误差
    
    def mv_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MV + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        
        def objective_function(weights):
            # 基本方差项
            variance_term = np.dot(weights, np.dot(cov_matrix, weights))
            # 预测收益项 (最大化预测收益，所以为负)
            expected_return_term = np.sum(weights * expected_returns)
            # 预测误差项 (最大化预测误差，所以为负)
            prediction_error_term = np.sum(weights * prediction_errors)
            
            # min variance - expected_return - prediction_error
            return variance_term - expected_return_term - prediction_error_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MV_Predictive')
    
    def msv_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSV + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        semivariance_matrix = self._calculate_semivariance_matrix(stock_codes, returns_data, cov_matrix)
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        
        def objective_function(weights):
            semivariance_term = np.dot(weights, np.dot(semivariance_matrix, weights))
            expected_return_term = np.sum(weights * expected_returns)
            prediction_error_term = np.sum(weights * prediction_errors)
            
            return semivariance_term - expected_return_term - prediction_error_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MSV_Predictive')
    
    def mad_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MAD + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def mad_predictive_linear_optimization():
            """MAD + 预测信息的线性规划求解"""
            
            # 目标函数系数
            c = np.zeros(n + T)
            c[:n] = -expected_returns - prediction_errors  # 预测信息项
            c[n:] = 1.0 / T  # MAD项
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            # 使用预测收益作为期望收益
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi
                constraint1 = np.zeros(n + T)
                constraint1[:n] = -deviation
                constraint1[n + t] = 1
                A_ub.append(constraint1)
                b_ub.append(0)
                
                # dt >= -Σ(r̂i - rit)*wi
                constraint2 = np.zeros(n + T)
                constraint2[:n] = deviation
                constraint2[n + t] = 1
                A_ub.append(constraint2)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        return mad_predictive_linear_optimization()
    
    def msad_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSAD + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def msad_predictive_linear_optimization():
            """MSAD + 预测信息的线性规划求解"""
            
            c = np.zeros(n + T)
            c[:n] = -expected_returns - prediction_errors
            c[n:] = 1.0 / T
            
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi (只考虑负偏差)
                constraint = np.zeros(n + T)
                constraint[:n] = -deviation
                constraint[n + t] = 1
                A_ub.append(constraint)
                b_ub.append(0)
            
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        
        return msad_predictive_linear_optimization()
    
    def cvar_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """CVaR + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        alpha = self.auxiliary_vars['alpha']
        
        def cvar_predictive_linear_optimization():
            """CVaR + 预测信息的线性规划求解"""
            
            c = np.zeros(n + 1 + T)
            c[:n] = -expected_returns - prediction_errors  # 预测信息项
            c[n] = 1.0  # ξ
            c[n+1:] = 1.0 / ((1 - alpha) * T)  # ηt
            
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                
                # ηt >= -Σwi*rit - ξ
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n] = 1
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(0)
            
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ξ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return cvar_predictive_linear_optimization()
    
    def omega_predictive_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """Omega + 预测信息模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        tau = self.auxiliary_vars['tau']
        delta = self.auxiliary_vars['delta']
        
        def omega_predictive_linear_optimization():
            """Omega + 预测信息的线性规划求解"""
            
            c = np.zeros(n + 1 + T)
            c[:n] = expected_returns + prediction_errors  # 预测信息项(取负)
            c[n] = -1.0  # -ψ (最大化ψ)
            
            A_ub = []
            b_ub = []
            
            # 使用预测收益作为期望收益
            main_constraint = np.zeros(n + 1 + T)
            main_constraint[:n] = -delta * expected_returns
            main_constraint[n] = 1  # ψ
            main_constraint[n+1:] = (1 - delta) / T
            A_ub.append(main_constraint)
            b_ub.append(-delta * tau)
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(tau)
            
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ψ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        return omega_predictive_linear_optimization()

    # ========== 4. 综合模型 (Integrated Models) ==========
    # 结合原始模型 + 高阶矩 + 预测信息
    
    def mv_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MV综合模型 - 原始+高阶矩+预测信息"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        
        def objective_function(weights):
            # 基本风险项
            variance_term = np.dot(weights, np.dot(cov_matrix, weights))
            # 预测信息项
            expected_return_term = np.sum(weights * expected_returns)
            prediction_error_term = np.sum(weights * prediction_errors)
            # 高阶矩项
            skewness_term = np.sum(weights * individual_skewness)
            kurtosis_term = np.sum(weights * individual_kurtosis)
            
            # min: variance - expected_return - prediction_error - skewness + kurtosis
            return variance_term - expected_return_term - prediction_error_term - skewness_term + kurtosis_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MV_Integrated')
    
    def msv_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSV综合模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        semivariance_matrix = self._calculate_semivariance_matrix(stock_codes, returns_data, cov_matrix)
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        
        def objective_function(weights):
            semivariance_term = np.dot(weights, np.dot(semivariance_matrix, weights))
            expected_return_term = np.sum(weights * expected_returns)
            prediction_error_term = np.sum(weights * prediction_errors)
            skewness_term = np.sum(weights * individual_skewness)
            kurtosis_term = np.sum(weights * individual_kurtosis)
            
            return semivariance_term - expected_return_term - prediction_error_term - skewness_term + kurtosis_term
        
        return self._solve_optimization_with_linear_constraints(objective_function, n, 'MSV_Integrated')
    
    def mad_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MAD综合模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def mad_integrated_linear_optimization():
            """MAD综合模型的线性规划求解"""
            
            # 目标函数系数
            c = np.zeros(n + T)
            # 预测信息 + 高阶矩项
            c[:n] = -expected_returns - prediction_errors - individual_skewness + individual_kurtosis
            c[n:] = 1.0 / T  # MAD项
            
            # 不等式约束
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi
                constraint1 = np.zeros(n + T)
                constraint1[:n] = -deviation
                constraint1[n + t] = 1
                A_ub.append(constraint1)
                b_ub.append(0)
                
                # dt >= -Σ(r̂i - rit)*wi
                constraint2 = np.zeros(n + T)
                constraint2[:n] = deviation
                constraint2[n + t] = 1
                A_ub.append(constraint2)
                b_ub.append(0)
            
            # 等式约束: Σwi = 1
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            # 变量边界
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return mad_integrated_linear_optimization()
    
    def msad_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """MSAD综合模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        
        def msad_integrated_linear_optimization():
            """MSAD综合模型的线性规划求解"""
            
            c = np.zeros(n + T)
            c[:n] = -expected_returns - prediction_errors - individual_skewness + individual_kurtosis
            c[n:] = 1.0 / T
            
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                deviation = expected_returns - returns_t
                
                # dt >= Σ(r̂i - rit)*wi (只考虑负偏差)
                constraint = np.zeros(n + T)
                constraint[:n] = -deviation
                constraint[n + t] = 1
                A_ub.append(constraint)
                b_ub.append(0)
            
            A_eq = np.zeros((1, n + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            for t in range(T):
                bounds.append((0, None))
            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        return msad_integrated_linear_optimization()
    
    def cvar_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """CVaR综合模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        alpha = self.auxiliary_vars['alpha']
        
        def cvar_integrated_linear_optimization():
            """CVaR综合模型的线性规划求解"""
            
            c = np.zeros(n + 1 + T)
            # 预测信息 + 高阶矩项
            c[:n] = -expected_returns - prediction_errors - individual_skewness + individual_kurtosis
            c[n] = 1.0  # ξ
            c[n+1:] = 1.0 / ((1 - alpha) * T)  # ηt
            
            A_ub = []
            b_ub = []
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                
                # ηt >= -Σwi*rit - ξ
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n] = 1
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(0)
            
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ξ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n

        return cvar_integrated_linear_optimization()
    
    def omega_integrated_optimization(self, stock_codes, returns_data, portfolio_stats, cov_matrix):
        """Omega综合模型"""
        n = len(stock_codes)
        if n == 0:
            return np.array([])
        
        expected_returns = np.array([portfolio_stats[code]['predicted_return'] for code in stock_codes])
        prediction_errors = np.array([portfolio_stats[code]['mean_prediction_error'] for code in stock_codes])
        individual_skewness = np.array([portfolio_stats[code]['skewness'] for code in stock_codes])
        individual_kurtosis = np.array([portfolio_stats[code]['kurtosis'] for code in stock_codes])
        min_length = min(len(returns_data[code]) for code in stock_codes)
        returns_matrix = np.array([returns_data[code][-min_length:] for code in stock_codes])
        T = returns_matrix.shape[1]
        tau = self.auxiliary_vars['tau']
        delta = self.auxiliary_vars['delta']
        
        def omega_integrated_linear_optimization():
            """Omega综合模型的线性规划求解"""
            
            c = np.zeros(n + 1 + T)
            # 预测信息 + 高阶矩项 (取负以最大化)
            c[:n] = expected_returns + prediction_errors + individual_skewness - individual_kurtosis
            c[n] = -1.0  # -ψ (最大化ψ)
            
            A_ub = []
            b_ub = []
            
            # 主约束，使用预测收益
            main_constraint = np.zeros(n + 1 + T)
            main_constraint[:n] = -delta * expected_returns
            main_constraint[n] = 1  # ψ
            main_constraint[n+1:] = (1 - delta) / T
            A_ub.append(main_constraint)
            b_ub.append(-delta * tau)
            
            for t in range(T):
                returns_t = returns_matrix[:, t]
                constraint = np.zeros(n + 1 + T)
                constraint[:n] = returns_t
                constraint[n + 1 + t] = -1
                A_ub.append(constraint)
                b_ub.append(tau)
            
            A_eq = np.zeros((1, n + 1 + T))
            A_eq[0, :n] = 1
            b_eq = np.array([1])
            
            bounds = []
            for i in range(n):
                bounds.append((self.min_weight, self.max_weight))
            bounds.append((None, None))  # ψ
            for t in range(T):
                bounds.append((0, None))  # ηt

            result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub),
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
            try:
                result = linprog(c, A_ub=np.array(A_ub), b_ub=np.array(b_ub), 
                               A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')
                
                if result.success:
                    return result.x[:n]  # 返回权重
                else:
                    return np.ones(n) / n
            except:
                return np.ones(n) / n
        
        return omega_integrated_linear_optimization()

    
    def _calculate_semivariance_matrix(self, stock_codes, returns_data, cov_matrix):
        """计算半方差矩阵 - 只考虑负收益的协方差"""
        n = len(stock_codes)
        semivariance_matrix = np.zeros((n, n))
        
        for i, code_i in enumerate(stock_codes):
            returns_i = returns_data[code_i]
            negative_returns_i = returns_i[returns_i < 0]
            
            for j, code_j in enumerate(stock_codes):
                returns_j = returns_data[code_j]
                negative_returns_j = returns_j[returns_j < 0]
                
                if len(negative_returns_i) > 1 and len(negative_returns_j) > 1:
                    min_len = min(len(negative_returns_i), len(negative_returns_j))
                    if min_len > 1:
                        semivariance_matrix[i, j] = np.cov(negative_returns_i[-min_len:], 
                                                         negative_returns_j[-min_len:])[0, 1]
                    else:
                        semivariance_matrix[i, j] = cov_matrix[i, j] * 0.5
                else:
                    semivariance_matrix[i, j] = cov_matrix[i, j] * 0.5
        
        # 确保半方差矩阵正定
        eigenvals = np.linalg.eigvals(semivariance_matrix)
        if np.min(eigenvals) <= 0:
            regularization = abs(np.min(eigenvals))
            semivariance_matrix += np.eye(n) * regularization
        
        return semivariance_matrix
    
    
    def daily_four_types_optimization(self):
        """每日四类投资组合模型优化"""
        
        # 获取所有交易日
        trading_dates = sorted(self.raw_data['date'].unique())
        
        # 为所有模型初始化结果字典
        daily_weights = {}
        for model_type in self.model_types:
            for model_name in self.model_types[model_type]:
                daily_weights[model_name] = {}
        
        # 优化方法映射
        optimization_methods = {
            # 原始模型
            'MV_Original': self.mv_original_optimization,
            'MSV_Original': self.msv_original_optimization,
            'MAD_Original': self.mad_original_optimization,
            'MSAD_Original': self.msad_original_optimization,
            'CVaR_Original': self.cvar_original_optimization,
            'Omega_Original': self.omega_original_optimization,
            
            # 高阶矩模型
            'MV_HigherMoments': self.mv_higher_moments_optimization,
            'MSV_HigherMoments': self.msv_higher_moments_optimization,
            'MAD_HigherMoments': self.mad_higher_moments_optimization,
            'MSAD_HigherMoments': self.msad_higher_moments_optimization,
            'CVaR_HigherMoments': self.cvar_higher_moments_optimization,
            'Omega_HigherMoments': self.omega_higher_moments_optimization,
            
            # 预测信息模型
            'MV_Predictive': self.mv_predictive_optimization,
            'MSV_Predictive': self.msv_predictive_optimization,
            'MAD_Predictive': self.mad_predictive_optimization,
            'MSAD_Predictive': self.msad_predictive_optimization,
            'CVaR_Predictive': self.cvar_predictive_optimization,
            'Omega_Predictive': self.omega_predictive_optimization,
            
            # 综合模型
            'MV_Integrated': self.mv_integrated_optimization,
            'MSV_Integrated': self.msv_integrated_optimization,
            'MAD_Integrated': self.mad_integrated_optimization,
            'MSAD_Integrated': self.msad_integrated_optimization,
            'CVaR_Integrated': self.cvar_integrated_optimization,
            'Omega_Integrated': self.omega_integrated_optimization,
        }
        
        total_dates = len(trading_dates)
        
        for i, date in enumerate(trading_dates):
            if i==1 or i==400 or i==800:
                print(f"处理日期: {date.strftime('%Y-%m-%d')} ({i}/{total_dates})")
            
            # 获取当日投资组合股票
            stock_codes = self.get_daily_portfolio_stocks(date)
            # 获取历史数据
            returns_data, portfolio_stats = self.get_historical_returns(stock_codes, date)

            cov_matrix = self.calculate_covariance_matrix(returns_data)
            
            # 对每个模型进行优化
            for model_name, optimization_method in optimization_methods.items():
                weights = optimization_method(stock_codes, returns_data, portfolio_stats, cov_matrix)

                daily_weights[model_name][date] = {
                    'stock_codes': stock_codes,
                    'weights': weights,
                    'method': model_name
                }
        
        self.daily_weights = daily_weights
        
        return self.daily_weights

    def export_daily_data(self, output_dir="."):
            # 四种扩展类型
            model_types = ['Original', 'HigherMoments', 'Predictive', 'Integrated']
            
            # 根据data_file生成文件名前缀
            prefix = ""
            if 'RF' in self.data_file:
                prefix += "RF-"
            if 'TI' in self.data_file:
                prefix += "TI-"
            if 'FI' in self.data_file:
                prefix += "FI-"
            
            # 根据交易成本率生成文件名后缀
            suffix = ""
            if abs(self.transaction_cost_rate - 0.001) < 1e-6:
                suffix = "0.1%"
            elif abs(self.transaction_cost_rate - 0.0005) < 1e-6:
                suffix = "0.05%"
            
            exported_files = []
            
            # 对每个基础模型分别导出
            for base_model in self.base_models:

                # 该基础模型的四种扩展模型名称
                model_names = [f'{base_model}_{model_type}' for model_type in model_types]
                
                # 收集所有日期
                all_dates = set()
                for model in model_names:
                    if model in self.portfolio_returns_with_cost:
                        dates = self.portfolio_returns_with_cost[model]['date']
                        all_dates.update(dates)
                
                sorted_dates = sorted(list(all_dates))
                export_data = {'date': sorted_dates}
                
                # 添加每个扩展类型的收益率和换手率
                for model_type in model_types:
                    model_name = f'{base_model}_{model_type}'
                    
                    if model_name in self.portfolio_returns_with_cost:
                        model_df = self.portfolio_returns_with_cost[model_name].set_index('date')
                        
                        # 为每个日期添加收益率和换手率数据
                        returns_col = f'{model_type}_return'
                        turnover_col = f'{model_type}_turnover'
                        
                        export_data[returns_col] = []
                        export_data[turnover_col] = []
                        
                        for date in sorted_dates:
                            if date in model_df.index:
                                # 净收益率
                                export_data[returns_col].append(model_df.loc[date, 'net_return'])
                                # 换手率
                                export_data[turnover_col].append(model_df.loc[date, 'turnover_rate'])

                export_df = pd.DataFrame(export_data)

                filename = f"{prefix}{base_model}四种模型每日收益率和换手率{suffix}.csv"
                filepath = f"{output_dir}/{filename}" if output_dir != "." else filename
                export_df.to_csv(filepath, index=False, encoding='utf-8-sig')
                
                exported_files.append(filename)
                print(f"已导出: {filename}") 
                print(f"列名: {list(export_df.columns)}")
            
            return exported_files

def main():
    optimizer = FourTypesPortfolioOptimizer(data_file)
    print("=== 四类投资组合模型优化过程===")
        
    # 1. 加载数据
    print("\n步骤1: 加载数据并分析每年股票")
    optimizer.load_data()

    # 2. 四类模型优化
    print("\n步骤2: 每日四类投资组合模型优化")
    optimizer.daily_four_types_optimization()
        
    # 3. 计算考虑交易成本的收益
    print("\n步骤3: 计算考虑交易成本的投资组合收益")
    optimizer.calculate_four_types_returns_with_cost()

    # 4. 导出数据
    print("\n步骤4: 导出数据")
    optimizer.export_daily_data()


LSTM （c = 0%）

In [7]:
data_file = "BigData_LSTM_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.000  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 21365 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-01-03 (1/1200)
处理日期: 2021-08-25 (400/1200)
处理日期: 2023-04-21 (800/1200)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: CVaR_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: Omega_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MV_HigherMome

RF （c = 0%）

In [8]:
data_file = "BigData_RF_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.000  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 21365 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-01-03 (1/1200)
处理日期: 2021-08-25 (400/1200)
处理日期: 2023-04-21 (800/1200)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: CVaR_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: Omega_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MV_HigherMome

LSTM-FI （c = 0%）

In [9]:
data_file = "FI_LSTM_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.000  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 1074 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-02-28 (1/60)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSV_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MAD_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSAD_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: CVaR_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: Omega_Original
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MV_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSV_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MAD_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSAD_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: CVaR_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: Omega_HigherMoments
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MV_Predictive
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSV_Predictive
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MAD_Predictive
  处理日期: 2020-02-28 00:00:00 (1/60)
处理模型: MSAD_Predictive
  处理日期: 2020

LSTM-TI （c = 0%）

In [10]:
data_file = "TI_LSTM_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.000  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 21606 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-01-03 (1/1200)
处理日期: 2021-08-25 (400/1200)
处理日期: 2023-04-21 (800/1200)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: CVaR_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: Omega_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MV_HigherMome

LSTM （c = 0.05%）

In [11]:
data_file = "BigData_LSTM_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.0005  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 21365 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-01-03 (1/1200)
处理日期: 2021-08-25 (400/1200)
处理日期: 2023-04-21 (800/1200)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: CVaR_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: Omega_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MV_HigherMome

LSTM （c = 0.1%）

In [12]:
data_file = "BigData_LSTM_predictions.csv"  # 数据文件路径
Transaction_cost_rate = 0.001  # 交易成本率
optimizer = main()

=== 四类投资组合模型优化过程===

步骤1: 加载数据并分析每年股票
总共 21365 条记录

步骤2: 每日四类投资组合模型优化
处理日期: 2020-01-03 (1/1200)
处理日期: 2021-08-25 (400/1200)
处理日期: 2023-04-21 (800/1200)

步骤3: 计算考虑交易成本的投资组合收益
处理模型: MV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSV_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MSAD_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: CVaR_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: Omega_Original
  处理日期: 2020-01-03 00:00:00 (1/1200)
  处理日期: 2021-08-25 00:00:00 (400/1200)
  处理日期: 2023-04-21 00:00:00 (800/1200)
处理模型: MV_HigherMome

In [13]:
import pandas as pd
def read_model_data(df,nxt):
    df['date'] = pd.to_datetime(df['date'])
    if nxt=="daily_returns":
        model_data = {
            'Original_return': df['Original_return'],
            'HigherMoments_return': df['HigherMoments_return'],
            'Predictive_return': df['Predictive_return'],
            'Integrated_return': df['Integrated_return']
        }
    elif nxt=="turnover_transformed":
        model_data = {
            'Original_turnover': df['Original_turnover'],
            'HigherMoments_turnover': df['HigherMoments_turnover'],
            'Predictive_turnover': df['Predictive_turnover'],
            'Integrated_turnover': df['Integrated_turnover']
        }
    return df['date'], model_data

def export_data_to_csv(all_dates, all_model_data, models_list, nxt):
    base_dates = all_dates[models_list[0]]
    returns_df = pd.DataFrame()
    returns_df['date'] = base_dates
    for model_name in models_list:
        model_data = all_model_data[model_name]
        if nxt=="daily_returns":
            returns_df[f'{model_name}_Original'] = model_data['Original_return'].values
            returns_df[f'{model_name}_HigherMoments'] = model_data['HigherMoments_return'].values
            returns_df[f'{model_name}_Predictive'] = model_data['Predictive_return'].values
            returns_df[f'{model_name}_Integrated'] = model_data['Integrated_return'].values
        elif nxt=="turnover_transformed":
            returns_df[f'{model_name}_Original_turnover'] = model_data['Original_turnover'].values
            returns_df[f'{model_name}_HigherMoments_turnover'] = model_data['HigherMoments_turnover'].values
            returns_df[f'{model_name}_Predictive_turnover'] = model_data['Predictive_turnover'].values
            returns_df[f'{model_name}_Integrated_turnover'] = model_data['Integrated_turnover'].values
    returns_df.to_csv(f'portfolio_{nxt}.csv', index=False, encoding='utf-8-sig')
    print(f"已导出 portfolio_{nxt}.csv")

models_list =['MV', 'MSV', 'MAD', 'MSAD', 'Omega', 'CVaR']
all_model_data = {}
all_dates = {}
nxt=["daily_returns","turnover_transformed"]
for nxt in nxt:
    for model_name in models_list:
        filename = f'{model_name}四种模型每日收益率和换手率.csv'
        df = pd.read_csv(filename)
        dates, model_data = read_model_data(df, nxt)
        all_dates[model_name] = dates
        all_model_data[model_name] = model_data
    export_data_to_csv(all_dates, all_model_data, models_list,nxt)

已导出 portfolio_daily_returns.csv
已导出 portfolio_turnover_transformed.csv


In [14]:
import pandas as pd
def read_model_data(df,nxt):
    df['date'] = pd.to_datetime(df['date'])
    if nxt=="daily_returns":
        model_data = {
            'Original_return': df['Original_return'],
            'HigherMoments_return': df['HigherMoments_return'],
            'Predictive_return': df['Predictive_return'],
            'Integrated_return': df['Integrated_return']
        }
    elif nxt=="turnover_transformed":
        model_data = {
            'Original_turnover': df['Original_turnover'],
            'HigherMoments_turnover': df['HigherMoments_turnover'],
            'Predictive_turnover': df['Predictive_turnover'],
            'Integrated_turnover': df['Integrated_turnover']
        }
    return df['date'], model_data

def export_data_to_csv(all_dates, all_model_data, models_list, nxt):
    base_dates = all_dates[models_list[0]]
    returns_df = pd.DataFrame()
    returns_df['date'] = base_dates
    for model_name in models_list:
        model_data = all_model_data[model_name]
        if nxt=="daily_returns":
            returns_df[f'{model_name}_Original'] = model_data['Original_return'].values
            returns_df[f'{model_name}_HigherMoments'] = model_data['HigherMoments_return'].values
            returns_df[f'{model_name}_Predictive'] = model_data['Predictive_return'].values
            returns_df[f'{model_name}_Integrated'] = model_data['Integrated_return'].values
        elif nxt=="turnover_transformed":
            returns_df[f'{model_name}_Original_turnover'] = model_data['Original_turnover'].values
            returns_df[f'{model_name}_HigherMoments_turnover'] = model_data['HigherMoments_turnover'].values
            returns_df[f'{model_name}_Predictive_turnover'] = model_data['Predictive_turnover'].values
            returns_df[f'{model_name}_Integrated_turnover'] = model_data['Integrated_turnover'].values
    returns_df.to_csv(f'rf_portfolio_{nxt}.csv', index=False, encoding='utf-8-sig')
    print(f"已导出 rf_portfolio_{nxt}.csv")

models_list =['MV', 'MSV', 'MAD', 'MSAD', 'Omega', 'CVaR']
all_model_data = {}
all_dates = {}
nxt=["daily_returns","turnover_transformed"]
for nxt in nxt:
    for model_name in models_list:
        filename = f'RF-{model_name}四种模型每日收益率和换手率.csv'
        df = pd.read_csv(filename)
        dates, model_data = read_model_data(df, nxt)
        all_dates[model_name] = dates
        all_model_data[model_name] = model_data
    export_data_to_csv(all_dates, all_model_data, models_list,nxt)

已导出 rf_portfolio_daily_returns.csv
已导出 rf_portfolio_turnover_transformed.csv


In [15]:
import pandas as pd
def read_model_data(df):
    df['date'] = pd.to_datetime(df['date'])
    model_data = {
        'Original_return': df['Original_return'],
        'HigherMoments_return': df['HigherMoments_return'],
        'Predictive_return': df['Predictive_return'],
        'Integrated_return': df['Integrated_return']
    }
    return df['date'], model_data

def export_data_to_csv(all_dates, all_model_data, models_list, nxt):
    base_dates = all_dates[models_list[0]]
    returns_df = pd.DataFrame()
    returns_df['date'] = base_dates
    for model_name in models_list:
        model_data = all_model_data[model_name]
        returns_df[f'{model_name}_Original'] = model_data['Original_return'].values
        returns_df[f'{model_name}_HigherMoments'] = model_data['HigherMoments_return'].values
        returns_df[f'{model_name}_Predictive'] = model_data['Predictive_return'].values
        returns_df[f'{model_name}_Integrated'] = model_data['Integrated_return'].values
    returns_df.to_csv(f'{nxt}_portfolio_daily_returns.csv', index=False, encoding='utf-8-sig')
    print(f"已导出 {nxt}_portfolio_daily_returns.csv")

models_list =['MV', 'MSV', 'MAD', 'MSAD', 'Omega', 'CVaR']
all_model_data = {}
all_dates = {}
nxt=["FI","TI"]
for nxt in nxt:
    for model_name in models_list:
        filename = f'{nxt}-{model_name}四种模型每日收益率和换手率.csv'
        df = pd.read_csv(filename)
        dates, model_data = read_model_data(df)
        all_dates[model_name] = dates
        all_model_data[model_name] = model_data
    export_data_to_csv(all_dates, all_model_data, models_list,nxt)

已导出 FI_portfolio_daily_returns.csv
已导出 TI_portfolio_daily_returns.csv


In [16]:
import pandas as pd
def read_model_data(df):
    df['date'] = pd.to_datetime(df['date'])
    model_data = {
        'Original_return': df['Original_return'],
        'HigherMoments_return': df['HigherMoments_return'],
        'Predictive_return': df['Predictive_return'],
        'Integrated_return': df['Integrated_return']
    }
    return df['date'], model_data

def export_data_to_csv(all_dates, all_model_data, models_list, nxt):
    base_dates = all_dates[models_list[0]]
    returns_df = pd.DataFrame()
    returns_df['date'] = base_dates
    for model_name in models_list:
        model_data = all_model_data[model_name]
        returns_df[f'{model_name}_Original'] = model_data['Original_return'].values
        returns_df[f'{model_name}_HigherMoments'] = model_data['HigherMoments_return'].values
        returns_df[f'{model_name}_Predictive'] = model_data['Predictive_return'].values
        returns_df[f'{model_name}_Integrated'] = model_data['Integrated_return'].values
    returns_df.to_csv(f'portfolio_daily_returns{nxt}.csv', index=False, encoding='utf-8-sig')
    print(f"已导出 portfolio_daily_returns{nxt}.csv")

models_list =['MV', 'MSV', 'MAD', 'MSAD', 'Omega', 'CVaR']
all_model_data = {}
all_dates = {}
nxt=["0.05","0.1"]
for nxt in nxt:
    for model_name in models_list:
        filename = f'{model_name}四种模型每日收益率和换手率{nxt}%.csv'
        df = pd.read_csv(filename)
        dates, model_data = read_model_data(df)
        all_dates[model_name] = dates
        all_model_data[model_name] = model_data
    export_data_to_csv(all_dates, all_model_data, models_list,nxt)

已导出 portfolio_daily_returns0.05.csv
已导出 portfolio_daily_returns0.1.csv
